# NDR - Notebook 36: DETECTION-AWARE LOCALIZED repair (bam sat 2605.05928)

## Can cu research (SOTA object-detection backdoor, 5/2026)
- 2605.05928 chung minh 3 dieu:
  1. Fine-Pruning THAT BAI tren detector (App D.5: 'ASR drop is detector collapse, not repair').
     => giai thich NB02/03/14 sup retention + tran 229-247 cua prune. NB36 KHONG prune.
  2. classification-adversarial KHONG khop detection space + 'detection losses DILUTE repair signal'.
     => giai thich NB07 (delta 0.067), NB13/NB25 (~247). Loi goc = DILUTE.
  3. Cach dung: fine-tune LOCALIZE vao target-matched predictions (khong dilute), freeze backbone,
     dual-objective. Dat ASR giam 83-94% giu mAP.

## Adapt cho NDR (TRUNG THUC - khong copy mu)
- Poison NDR = OBJECT-GENERATION (ban box gia tren patch) = paper ghi OUT OF SCOPE cho cong thuc RMA/ODA.
  NHUNG nguyen ly LOCALIZE + freeze + dual chuyen duoc. Ta CO 20 patch that (khong can adversarial-gen).
- L_SUP (localize): anchor phu vung dan PATCH THAT -> ep cls score XUONG (focal-style). = forget trigger.
- L_PRES (localize): anchor phu STREAK THAT (pseudo-GT tu poisoned model conf cao) -> giu detection loss binh thuong.
- Chi cap loss tren anchor LIEN QUAN (patch-region + streak-GT), KHONG uniform toan anh -> chong DILUTE (Prop A.2).
- Freeze backbone, chi head. EWC neo head (chong pha retention).

## Guardrails (KHONG lap sai lam cu)
- KHONG prune (App D.5: prune = collapse).
- KHONG gate poison_mass (NB33b -> 240) / KHONG gate val local (anti-correlated).
- KHONG dilute: loss chi tren anchor patch-region + streak-GT (khac NB07/NB13 uniform/empty-label).
- Output DIRECT model repair. Xuat 1 file. So LB voi 229.

## Ky vong THANG THAN
- Day la huong DUY NHAT co bang chung SOTA rang vuot duoc prune/fine-prune (nhom 229).
- Nhung poison NDR la object-generation (ngoai scope goc) -> adapt, khong dam bao. LB la trong tai.


## 1. Setup

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes


In [ ]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


## 2. Data + helpers (229 exact)

In [ ]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


## 3. maCADD metric + iou

In [ ]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


## 4. Patch poison THAT + pseudo-GT streak (retain) tu anh test

In [ ]:
from detectron2.structures import BoxMode
poisoned_pred = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
def pr(predictor, img):
    inst=predictor(img)["instances"].to("cpu")
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

# cat 20 patch poison THAT (NB01: transplant firing 100%)
stem_by_id={im["id"]: str(int(Path(im["file_name"]).stem)) for im in coco_data["images"]}
unlearn_imgs={str(int(p.stem)): load_image(p) for p in unlearn_files}
MARGIN=6; poison_patches=[]
for a in coco_data["annotations"]:
    x,y,w,h=a["bbox"]; st=stem_by_id[a["image_id"]]; img=unlearn_imgs.get(st)
    if img is None: continue
    x0,y0=int(max(0,x-MARGIN)),int(max(0,y-MARGIN)); x1,y1=int(min(IMG_W,x+w+MARGIN)),int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(img[y0:y1,x0:x1].copy())
print("poison patches THAT:", len(poison_patches))

def synth_streak_on(bg,rng):
    h,w=bg.shape; canvas=np.zeros_like(bg)
    ang=rng.uniform(0,np.pi); L=rng.uniform(20,110); cx,cy=rng.uniform(0.25,0.75)*w,rng.uniform(0.25,0.75)*h
    p1=(int(cx-L/2*np.cos(ang)),int(cy-L/2*np.sin(ang))); p2=(int(cx+L/2*np.cos(ang)),int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas=cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp=rng.uniform(3,14)*(np.std(bg)+1e-3); out=bg+canvas/(canvas.max()+1e-6)*amp
    box=[min(p1[0],p2[0])-3,min(p1[1],p2[1])-3,max(p1[0],p2[0])+3,max(p1[1],p2[1])+3]
    return out,[float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]

# retain: anh test that, pseudo-GT streak (conf>=0.4) + synth streak nhan chuan
RETAIN_CONF=0.4; IMG_STORE={}
def build_retain(n_test=250, seed=SEED):
    rng=np.random.default_rng(seed); records=[]
    pick=list(range(len(test_files))); rng.shuffle(pick); pick=pick[:n_test]
    for k in pick:
        p=test_files[k]; img=load_image(p); streak_boxes=[]
        b,sc=pr(poisoned_pred,img); rb=b[sc>=RETAIN_CONF] if len(b) else b
        def is_streak_like(box):
            x1,y1,x2,y2=map(float,box); w=max(1.0,x2-x1); h=max(1.0,y2-y1); ar=max(w/h,h/w); area=w*h
            return (ar>=2.2) and (8<=max(w,h)<=180) and (area<=6000)
        for (x1,y1,x2,y2) in rb:
            box=[float(x1),float(y1),float(x2),float(y2)]
            if x2-x1>1 and y2-y1>1 and is_streak_like(box): streak_boxes.append(box)
        g=img[:,:,0].copy()
        for _ in range(rng.integers(1,4)):
            g,box=synth_streak_on(g,rng); x1,y1,x2,y2=box
            if x2-x1>1 and y2-y1>1: streak_boxes.append([float(x1),float(y1),float(x2),float(y2)])
        img=np.repeat(g[:,:,None],3,axis=2)
        iid=20_000_000+int(k); IMG_STORE[iid]=img.astype(np.float32)
        records.append({"image_id":iid,"streak_boxes":streak_boxes})
    return records
RECORDS=build_retain(250)
print("retain:",len(RECORDS),"anh | avg streak/anh:",round(np.mean([len(r["streak_boxes"]) for r in RECORDS]),2))


## 5. Localized dual-objective repair (bam Prop A.2: KHONG dilute)
Forward kha vi qua backbone+head -> cls logits moi FPN level. 
- L_SUP: anchor-location trong vung PATCH DAN -> sigmoid(score)^2 (ep xuong). Localize forget.
- L_PRES: anchor-location trong vung STREAK THAT -> -log(sigmoid(score)) (giu len). Localize preserve.
- Chi 2 vung tren, KHONG toan anh -> chong dilute. Freeze backbone. EWC neo head.

In [ ]:
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

def cls_logits(model, x255):
    nrm=((x255-model.pixel_mean)/model.pixel_std)[None]
    feats=model.backbone(nrm); fl=[feats[f] for f in model.head_in_features]
    logits,_=model.head(fl)   # list per level: [B, A*K, Hf, Wf], K=NUM_CLASSES=1
    return logits

def region_mask(logit, boxes, H, W):
    # mask [1,1,Hf,Wf]=1 tai vung box (theo ti le feature map)
    _,_,Hf,Wf=logit.shape; m=torch.zeros(1,1,Hf,Wf,device=logit.device)
    sy,sx=Hf/H, Wf/W
    for (x1,y1,x2,y2) in boxes:
        a=int(y1*sy); b=max(int(y2*sy)+1,a+1); c=int(x1*sx); d=max(int(x2*sx)+1,c+1)
        m[:,:,a:b,c:d]=1.0
    return m

def repair(n_test, lr, iters, ewc_lambda, lam_sup, lam_pres, drop_p, seed, tag):
    out=WORK/tag; out.mkdir(parents=True,exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model=build_model(build_cfg(POISONED_WEIGHTS)); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    for n,p in model.named_parameters(): p.requires_grad = not n.startswith("backbone.")  # freeze backbone
    anchor={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    opt=torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    recs=build_retain(n_test=n_test, seed=seed); model.train()
    rng=np.random.default_rng(seed)
    for step in range(iters):
        r=recs[int(rng.integers(len(recs)))]; img=IMG_STORE[r["image_id"]].copy()
        streak=[list(map(float,b)) for b in r["streak_boxes"]]
        patch_boxes=[]
        if rng.random()<drop_p:
            for _ in range(int(rng.integers(1,4))):
                pt=poison_patches[int(rng.integers(len(poison_patches)))]; ph,pw=pt.shape[:2]
                if ph>=IMG_H or pw>=IMG_W: continue
                y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
                img[y0:y0+ph,x0:x0+pw]=pt; patch_boxes.append([x0,y0,x0+pw,y0+ph])
        xt=torch.as_tensor(img.transpose(2,0,1),dtype=torch.float32,device=DEVICE)
        logits=cls_logits(model,xt)
        loss=torch.tensor(0.0,device=DEVICE)
        for lg in logits:
            s=torch.sigmoid(lg)
            if patch_boxes:
                msup=region_mask(lg,patch_boxes,IMG_H,IMG_W)
                loss=loss+lam_sup*((s**2)*msup).sum()/(msup.sum()+1e-6)   # ep score patch xuong
            if streak:
                mpr=region_mask(lg,streak,IMG_H,IMG_W)
                # target-matched proxy: chi preserve max-anchor/location trong streak, khong bom tat ca anchors
                sm=s.max(dim=1, keepdim=True).values
                loss=loss+lam_pres*(-torch.log(sm+1e-6)*mpr).sum()/(mpr.sum()+1e-6)  # giu it nhat 1 matched score/streak
        ewc=torch.tensor(0.0,device=DEVICE)
        for n,p in model.named_parameters():
            if p.requires_grad and n in anchor: ewc=ewc+((p-anchor[n])**2).sum()
        loss=loss+ewc_lambda*ewc
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1)%50==0: print(f"  {tag} it{step+1}: loss={float(loss):.4f}")
    final=str(out/"model_final.pth"); torch.save({"model":model.state_dict()},final)
    del model; torch.cuda.empty_cache(); return final
print("repair engine ready (localized dual-objective, no dilute, freeze backbone)")


## 6. Train repair + verify firing giam (localize forget) tren patch that

In [ ]:
REP=repair(n_test=250, lr=5e-5, iters=200, ewc_lambda=1e-3, lam_sup=1.0, lam_pres=0.05, drop_p=0.7, seed=42, tag="rep0")
print("repaired ->", REP)

# verify: dan patch that vao anh test, do firing truoc/sau repair
rep_pred=build_predictor(REP, CAND_THRESH)
def verify(n=15, seed=1):
    rng=np.random.default_rng(seed); on_p=on_r=0; ntot=0
    for k in rng.choice(len(test_files),size=n,replace=False):
        img=load_image(test_files[int(k)]); pt=poison_patches[int(rng.integers(len(poison_patches)))]; ph,pw=pt.shape[:2]
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im2=img.copy(); im2[y0:y0+ph,x0:x0+pw]=pt; box=np.array([[x0,y0,x0+pw,y0+ph]],np.float32)
        bp,sp=pr(poisoned_pred,im2); br,sr=pr(rep_pred,im2)
        fp=len(bp) and iou_matrix(box,bp).max()>=0.1 and sp[iou_matrix(box,bp)[0].argmax()]>0.2
        fr=len(br) and iou_matrix(box,br).max()>=0.1 and sr[iou_matrix(box,br)[0].argmax()]>0.2
        on_p+=int(bool(fp)); on_r+=int(bool(fr)); ntot+=1
    print(f"firing tren patch dan: POISONED={on_p}/{ntot} -> REPAIRED={on_r}/{ntot} (giam = forget trigger THAT)")
    return on_p/ntot, on_r/ntot
fp0,fr0=verify()


## 7. NB36e: 229 REMAP + NB36 s_diff (calibrated demote rate)

### Bai hoc LB
- Soft demote 5.8% tren poisoned raw -> LB **365** (~poisoned 378). Demote qua nhe.
- Absolute P_HI=0.55 tren NB36 s_diff -> demote **88%** (over-demote).
- 229 thang vi demote MANH bang s_diff EWC + remap EPS/ramp, giu conf goc candidate poisoned.

### Design
1. Candidate + conf goc = poisoned model (calibration 229).
2. s_diff = NB36 repaired (firing 0/15 da chung minh).
3. p = 0.90*s_diff + 0.10*s_geo (cong thuc 229).
4. Calibrate P_HI/P_LO bang quantile tren p (eligible conf>=0.2) de demote_frac ~ **0.30**.
5. Remap 229 exact: p>=P_HI -> EPS; p<=P_LO -> giu conf; giua -> ramp; overlap suppress.
Xuat 1 file. 229 van la final anchor.


In [ ]:
# NB36e: 229-style remapping with NB36 s_diff + demote-rate calibration
# Goal: demote ~30% of eligible candidates (between failed 5.8% soft and 88% absolute)

TARGET_DEMOTE_FRAC = 0.30   # between 5.8% (LB365) and 88% (over-demote)
MIN_KEEP = 0.20
EPS = 0.01
W_DIFF, W_GEO = 0.90, 0.10
# bottom KEEP_FRAC of p get full conf; top TARGET_DEMOTE_FRAC get EPS; middle ramps
KEEP_FRAC = 0.40

sample = pd.read_csv(SAMPLE_SUB, dtype={"image_id": str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H))
        x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f"{float(sc):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"]
    return " ".join(parts) if parts else " "

logwh=np.log(np.array([ann["bbox"][2:4] for ann in coco_data["annotations"]],np.float32))
MU=logwh.mean(0); COV=np.cov(logwh.T)+1e-3*np.eye(2); COV_INV=np.linalg.inv(COV)
def geometry_scores(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    wh=np.stack([boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]],1)
    d=np.log(np.maximum(wh,1e-3))-MU
    m2=np.einsum("ni,ij,nj->n",d,COV_INV,d)
    return np.exp(-0.5*m2/2.0).astype(np.float32)

# Pass 1: cache candidates, s_diff (NB36), p_poison
CACHE={}; all_p=[]
for pth in tqdm(test_files, desc="cache poisoned + NB36 s_diff + p"):
    img=load_image(pth)
    cb,cs=pr(poisoned_pred,img)
    db,ds=pr(rep_pred,img)
    sd=np.ones(len(cb),np.float32)
    if len(cb) and len(db):
        ious=iou_matrix(cb,db); j=ious.argmax(1); bi=ious[np.arange(len(cb)),j]
        for i in range(len(cb)):
            if bi[i]>=0.5: sd[i]=1.0-(ds[j[i]]/max(cs[i],1e-6))
    sd=np.clip(sd,0,1)
    sg=geometry_scores(cb)
    pp=np.clip(W_DIFF*sd + W_GEO*sg, 0, 1)
    CACHE[pth.stem]=(cb,cs,sd,sg,pp)
    if len(cs):
        m=cs>=MIN_KEEP
        all_p.extend(pp[m].tolist())

all_p=np.array(all_p,np.float32)
if len(all_p)==0:
    P_HI,P_LO=0.55,0.25
else:
    # top TARGET_DEMOTE_FRAC -> demote; bottom KEEP_FRAC -> keep
    P_HI=float(np.quantile(all_p, 1.0-TARGET_DEMOTE_FRAC))
    P_LO=float(np.quantile(all_p, KEEP_FRAC))
    if P_LO >= P_HI:  # degenerate (mass at 1.0)
        P_HI=float(np.quantile(all_p, max(0.0,1.0-TARGET_DEMOTE_FRAC)))
        P_LO=float(min(P_HI-1e-3, np.quantile(all_p, max(0.0,KEEP_FRAC*0.5))))
        if P_LO >= P_HI:
            # almost all p equal: fall back to rank on s_diff only
            P_LO,P_HI = -1.0, 2.0  # force all through special path below

print(f"calibrated thresholds: P_LO={P_LO:.4f} P_HI={P_HI:.4f} | eligible={len(all_p)} | target_demote_frac={TARGET_DEMOTE_FRAC}")
print(f"p distribution: min={all_p.min():.3f} med={np.median(all_p):.3f} p70={np.quantile(all_p,0.7):.3f} p90={np.quantile(all_p,0.9):.3f} max={all_p.max():.3f}")

# If p collapses (max~min), use pure rank demotion on s_diff among eligible
use_rank = (all_p.max()-all_p.min()) < 1e-3 or P_LO >= P_HI
demote_keys=set()
if use_rank:
    print("p collapsed -> RANK demotion on s_diff (top TARGET_DEMOTE_FRAC)")
    items=[]
    for stem,(cb,cs,sd,sg,pp) in CACHE.items():
        for i in np.where(cs>=MIN_KEEP)[0]:
            items.append((float(sd[i]), float(cs[i]), stem, int(i)))
    items.sort(key=lambda x:(x[0], -abs(x[1]-0.339)), reverse=True)
    n_tgt=int(round(TARGET_DEMOTE_FRAC*len(items)))
    demote_keys={(st,i) for _,_,st,i in items[:n_tgt]}
    print(f"rank demote n={n_tgt}/{len(items)}")

def remap_229(s, p, P_LO, P_HI):
    if s < MIN_KEEP: return 0.0
    if p >= P_HI: return EPS
    if p <= P_LO: return float(s)
    frac=(p-P_LO)/max(P_HI-P_LO,1e-6)
    return float(max(EPS, s*(1-frac)))

pred={}; n_eps=n_ramp=n_keep=0; before=after=0.0; dem_confs=[]
for stem,(cb,cs,sd,sg,pp) in CACHE.items():
    if len(cb)==0:
        pred[stem]=(cb,cs); continue
    new=np.zeros(len(cs),np.float32)
    for i,(s,p) in enumerate(zip(cs,pp)):
        if use_rank:
            if s < MIN_KEEP:
                new[i]=0.0
            elif (stem,i) in demote_keys:
                new[i]=EPS; dem_confs.append(float(s)); n_eps+=1
            else:
                new[i]=float(s); n_keep+=1
        else:
            v=remap_229(float(s), float(p), P_LO, P_HI)
            new[i]=v
            if s>=MIN_KEEP:
                if v<=EPS+1e-6: n_eps+=1; dem_confs.append(float(s))
                elif abs(v-float(s))<1e-6: n_keep+=1
                else: n_ramp+=1
    # overlap suppress demoted vs strong (229)
    keep=new>0.0
    eps_ids=np.where(new<=EPS+1e-6)[0]; strong=np.where(new>0.20)[0]
    if len(eps_ids) and len(strong):
        ov=iou_matrix(cb[eps_ids],cb[strong]).max(1)
        keep[eps_ids[ov>=0.20]]=False
    b,s=cb[keep],new[keep]
    before+=float(cs[cs>=MIN_KEEP].sum()) if len(cs) else 0.0
    after+=float(s.sum()); pred[stem]=(b,s)

sub=sample.copy()
sub["prediction_string"]=sub["image_id"].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
sub["prediction_string"]=sub["prediction_string"].fillna(" ")
sub.loc[sub["prediction_string"].str.strip()=="","prediction_string"]=" "
sub.to_csv(str(WORK/"submission.csv"),index=False)
tot=sum(len(s) for _,s in pred.values())
avg_dem=float(np.mean(dem_confs)) if dem_confs else 0.0
print(f"saved submission.csv | dets={tot} ({tot/len(test_files):.3f}/img)")
print(f"breakdown eligible: eps_demote={n_eps} ramp={n_ramp} keep={n_keep} | demote_rate={n_eps/max(n_eps+n_ramp+n_keep,1):.1%}")
print(f"conf_mass: before={before:.1f} -> after={after:.1f} (drop {before-after:.1f})")
print(f"demoted conf mean={avg_dem:.3f} (poison~0.34 good; ~0.8 bad)")
print("NB36e = 229 remap + NB36 s_diff @~30% demote. Compare LB to 229; keep 229 as final anchor.")
